# Data Science Internship – February 2026## Task 3: Build a Chatbot using Hugging Face Transformers

### What this notebook does1. Loads a pre-trained conversational transformer model from Hugging Face.2. Accepts user messages (console-based).3. Generates responses using transformer text generation (no hardcoded replies).4. Maintains conversation context across turns.5. Stops when the user types `exit` or `quit`.

In [ ]:
import torchfrom transformers import AutoTokenizer, AutoModelForCausalLM# Small conversational model for faster downloads.MODEL_NAME = "microsoft/DialoGPT-small"device = torch.device("cuda" if torch.cuda.is_available() else "cpu")tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)model.to(device)print("Model loaded:", MODEL_NAME)print("Device:", device)

In [ ]:
def generate_chat_response(user_message, chat_history_ids=None, max_new_tokens=60):    """Generate a chatbot response for one user turn.    Returns:        (bot_reply, updated_chat_history_ids)    """    if user_message is None:        user_message = ""    user_message = str(user_message).strip()    if not user_message:        return "", chat_history_ids    # DialoGPT expects each message to end with an EOS token.    new_user_input_ids = tokenizer.encode(        user_message + tokenizer.eos_token,        return_tensors="pt",    ).to(device)    # Add current user turn to the previous conversation tokens.    if chat_history_ids is not None:        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)    else:        bot_input_ids = new_user_input_ids    # Generate a response using prompt-based text generation.    output_ids = model.generate(        bot_input_ids,        max_new_tokens=max_new_tokens,        do_sample=True,        temperature=0.7,        top_p=0.9,        pad_token_id=tokenizer.eos_token_id,    )    # Only decode the newly generated part (not the prompt).    response_ids = output_ids[:, bot_input_ids.shape[-1]:]    bot_reply = tokenizer.decode(response_ids[0], skip_special_tokens=True).strip()    return bot_reply, output_ids

In [ ]:
def chat_interactive():    """Console-based chatbot loop."""    print("Hello! I am your AI assistant. How can I help you today?")    print("Type 'exit' or 'quit' to stop.\n")    chat_history_ids = None    while True:        user_input = input("You: ").strip()        if not user_input:            continue        if user_input.lower() in {"exit", "quit"}:            print("Bot: Goodbye! Take care.")            break        bot_reply, chat_history_ids = generate_chat_response(            user_input,            chat_history_ids=chat_history_ids,            max_new_tokens=60,        )        if bot_reply:            print("Bot:", bot_reply)        else:            print("Bot: (no reply generated)")# Uncomment the next line to run the interactive console chatbot.# chat_interactive()

In [ ]:
# Demo chat to show chatbot interaction outputs in the notebook.demo_prompts = [    "Hello",    "What is Artificial Intelligence?",    "Who created Python?",    "Thank you",]print("---- Demo Chat ----")chat_history_ids = Nonefor msg in demo_prompts:    bot_reply, chat_history_ids = generate_chat_response(        msg,        chat_history_ids=chat_history_ids,        max_new_tokens=60,    )    print("You:", msg)    print("Bot:", bot_reply if bot_reply else "(no reply generated)")    print("-")

## LinkedIn Post (copy-paste)**Summary:**During this NLP task, I built a console chatbot using Hugging Face Transformers and a pre-trained conversational model. I learned how transformer-based text generation can be used to generate context-aware responses.**Key Learnings:**- Loading and using pre-trained models from Hugging Face Model Hub- Tokenization and EOS handling for dialogue turns- Text generation using `model.generate()`- Maintaining conversation history across turns- Building an interactive chatbot loop with exit conditions**Acknowledgment:**Thanks to Innomatics Research Labs, my trainer, and my mentor for the guidance and support throughout this internship.**Hashtags:** #NLP #AI #DataScience #MachineLearning